In [ ]:
import pandas                             as      pd
import numpy                              as      np
import matplotlib.pyplot                  as      plt
import seaborn                            as      sns
from   IPython.display                    import  display
from   pylab                              import  rcParams 
from   datetime                           import  datetime, timedelta
from statsmodels.tsa.stattools            import  adfuller
from statsmodels.tsa.stattools            import  pacf
from statsmodels.tsa.stattools            import  acf
from statsmodels.graphics.tsaplots        import  plot_pacf
from statsmodels.graphics.tsaplots        import  plot_acf
from statsmodels.graphics.gofplots        import  qqplot
from statsmodels.tsa.seasonal             import  seasonal_decompose
from statsmodels.tsa.arima.model          import  ARIMA
from statsmodels.tsa.statespace.sarimax   import  SARIMAX
from statsmodels.tsa.api                  import  ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
df = pd.read_csv('train_(1).csv')

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

converting data into time series

In [ ]:
date = pd.date_range(start = '01/01/1749', end = '12/01/2010', freq = 'D')
date

In [ ]:
df['Time_Stamp'] = pd.DataFrame(date)

In [ ]:
df = df.set_index('Time_Stamp')

In [ ]:
df.head()

In [ ]:
plt.rcParams['figure.figsize'] = 20,10
df['Avg_sunspot_count'].plot()
plt.title('Monthly Average Sunspot Count (1749-2010)')
plt.xlabel('Year')
plt.ylabel('Average Sunspot Count')
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop(['Month'],axis=1, inplace = True)

In [ ]:
decomposition = seasonal_decompose(df, model = 'additive',period=7)
decomposition.plot()
plt.show()

In [ ]:
monthly_mean = df.resample('M').mean()
monthly_mean.plot.bar(figsize=(15, 8))
plt.show()

Variation in monthly mean plot is indicating that series is non-stationary

ACF and PACF plots for the series

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
plot_acf(df, lags=120, ax=ax);
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
plot_pacf(df, lags=120, ax=ax);
plt.show()

In [ ]:

train = df['Avg_sunspot_count']
test = pd.read_csv('test_(1).csv')

In [ ]:
dfObj = pd.DataFrame(columns= ['param', 'seasonal', 'AIC'])
dfObj

In [ ]:
#df['log_count'] = np.log1p(df['Avg_sunspot_count'])
df['log_count'] = np.log(df['Avg_sunspot_count'].replace(0, 1))  # Avoid log(0)

In [ ]:
import pandas as pd
import itertools
import statsmodels.api as sm

# Parameter grids
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in pdq]  # Monthly seasonality

dfObj = pd.DataFrame(columns=['param', 'seasonal', 'AIC'])

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            mod = sm.tsa.statespace.SARIMAX(df['log_count'],
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            results_SARIMAX = mod.fit(disp=False)
            dfObj = dfObj._append({'param': param, 'seasonal': param_seasonal, 'AIC': results_SARIMAX.aic}, ignore_index=True)
            print(f'SARIMA{param}x{param_seasonal} - AIC: {results_SARIMAX.aic:.2f}')
        except Exception as e:
            print(f'SARIMA{param}x{param_seasonal} failed: {e}')

In [ ]:
model = SARIMAX(df['log_count'],
                order=(1,1,1),
                seasonal_order=(1,1,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)

results = model.fit(disp=False)

In [ ]:
n_steps = len(test)
forecast = results.get_forecast(steps=120)
forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

In [ ]:
forecast_df = pd.DataFrame({
    'Month': test['Month'],
    'Avg_sunspot_count': forecast_mean.values.round()
})


In [ ]:
forecast_df

In [ ]:
forecast_df.to_csv('submission.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
df['log_count'].plot(title='Log1p Transformed Sunspot Count')
plt.show()

In [ ]:
pip install prophet

In [23]:
# rmse is 22.7
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from prophet import Prophet
from sklearn.metrics import mean_squared_error
from math import sqrt

# ---------------------- 1. Load Data ----------------------
train = pd.read_csv("train_(1).csv")
sample = pd.read_csv("sample_submission_(1).csv")

train['Month'] = pd.to_datetime(train['Month'], format='%d-%m-%Y')
sample['Month'] = pd.to_datetime(sample['Month'], format='%d-%m-%Y')

train = train[train['Month'].dt.year >= 1900].copy()
train['y'] = np.log1p(train['Avg_sunspot_count'])

# ---------------------- 2. Feature Engineering ----------------------
def create_features(df):
    df['month'] = df['Month'].dt.month
    df['year'] = df['Month'].dt.year
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
    df['cycle_phase'] = ((df['year'] - 1900) % 11) / 11
    df['fourier1'] = np.sin(2 * np.pi * df['Month'].dt.dayofyear / 132)
    df['fourier2'] = np.cos(2 * np.pi * df['Month'].dt.dayofyear / 132)
    df['lag12'] = df['y'].shift(12)
    df['lag24'] = df['y'].shift(24)
    df['lag36'] = df['y'].shift(36)
    df['rolling12'] = df['y'].rolling(12).mean().shift(1)
    df['rolling24'] = df['y'].rolling(24).mean().shift(1)
    df['delta'] = df['y'].diff().shift(1)
    return df

train = create_features(train).dropna().reset_index(drop=True)

feature_cols = ['sin_month', 'cos_month', 'cycle_phase', 'year',
                'lag12', 'lag24', 'lag36', 'rolling12', 'rolling24',
                'fourier1', 'fourier2', 'delta']

# ---------------------- 3. XGBoost Training ----------------------
X = train[feature_cols]
y = train['y']

xgb_model = XGBRegressor(
    n_estimators=600, learning_rate=0.03,
    max_depth=4, subsample=0.9, colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X, y)

# ---------------------- 4. Prophet Model ----------------------
prophet_df = train[['Month', 'y']].rename(columns={'Month': 'ds'})
prophet_df['cycle_phase'] = train['cycle_phase']
prophet_df['sin_month'] = train['sin_month']

prophet = Prophet()
prophet.add_regressor('cycle_phase')
prophet.add_regressor('sin_month')
prophet.fit(prophet_df)

# ---------------------- 5. Validation Split (2000–2010) ----------------------
val_data = train[train['Month'].dt.year >= 2000].copy()
val_true = np.expm1(val_data['y'])

# XGBoost validation predictions
val_xgb = np.expm1(xgb_model.predict(val_data[feature_cols]))

# Prophet validation predictions
prophet_train = train[train['Month'].dt.year < 2000].copy()
prophet_train_df = prophet_train[['Month', 'y']].rename(columns={'Month': 'ds'})
prophet_train_df['cycle_phase'] = prophet_train['cycle_phase']
prophet_train_df['sin_month'] = prophet_train['sin_month']

val_model = Prophet()
val_model.add_regressor('cycle_phase')
val_model.add_regressor('sin_month')
val_model.fit(prophet_train_df)

val_cat = np.expm1(cat_model.predict(X_val))

# 2. Prophet validation predictions using the SAME dates as val_data
val_future = pd.DataFrame({
    'ds': val_data['Month'],
    'cycle_phase': val_data['cycle_phase'],
    'sin_month': val_data['sin_month']
})
val_prophet_forecast = prophet.predict(val_future)
val_prophet = np.expm1(val_prophet_forecast['yhat'])

# ---------------------- 6. Tune Ensemble Weights ----------------------
best_rmse = float('inf')
best_w = 0.5

for w in np.linspace(0.1, 0.9, 9):
    blended = w * np.array(val_prophet) + (1 - w) * np.array(val_xgb)
    rmse = sqrt(mean_squared_error(val_true, blended))
    if rmse < best_rmse:
        best_rmse = rmse
        best_w = w

print(f"✅ Best Validation RMSE: {best_rmse:.2f} at Ensemble Weight: {best_w}")

# ---------------------- 7. Forecasting (Step-by-step) ----------------------
history = train.copy()
xgb_preds = []

for date in sample['Month']:
    row = {}
    row['month'] = date.month
    row['year'] = date.year
    row['sin_month'] = np.sin(2 * np.pi * row['month'] / 12)
    row['cos_month'] = np.cos(2 * np.pi * row['month'] / 12)
    row['cycle_phase'] = ((row['year'] - 1900) % 11) / 11
    row['fourier1'] = np.sin(2 * np.pi * date.dayofyear / 132)
    row['fourier2'] = np.cos(2 * np.pi * date.dayofyear / 132)
    row['lag12'] = history.iloc[-12]['y']
    row['lag24'] = history.iloc[-24]['y']
    row['lag36'] = history.iloc[-36]['y']
    row['rolling12'] = history['y'].iloc[-12:].mean()
    row['rolling24'] = history['y'].iloc[-24:].mean()
    row['delta'] = history['y'].diff().iloc[-1]

    X_pred = pd.DataFrame([row])[feature_cols]
    log_pred = xgb_model.predict(X_pred)[0]
    y_pred = np.expm1(log_pred)
    xgb_preds.append(y_pred)

    row['Month'] = date
    row['y'] = log_pred
    history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)

# ---------------------- 8. Prophet Final Forecast ----------------------
future_df = pd.DataFrame({
    'ds': sample['Month'],
    'cycle_phase': ((sample['Month'].dt.year - 1900) % 11) / 11,
    'sin_month': np.sin(2 * np.pi * sample['Month'].dt.month / 12)
})
prophet_preds = np.expm1(prophet.predict(future_df)['yhat']).clip(lower=0)

# ---------------------- 9. Final Ensemble + Submission ----------------------
final_preds = best_w * np.array(prophet_preds) + (1 - best_w) * np.array(xgb_preds)

submission = pd.DataFrame({
    'Month': sample['Month'].dt.strftime('%d-%m-%Y'),
    'Avg_sunspot_count': np.round(final_preds)
})
submission.to_csv("submission_final.csv", index=False)
print("📁 submission_final.csv saved ✅")

20:36:06 - cmdstanpy - INFO - Chain [1] start processing
20:36:06 - cmdstanpy - INFO - Chain [1] done processing
20:36:06 - cmdstanpy - INFO - Chain [1] start processing
20:36:06 - cmdstanpy - INFO - Chain [1] done processing


✅ Best Validation RMSE: 17.34 at Ensemble Weight: 0.1
📁 submission_final.csv saved ✅


In [24]:
submission

,Month,Avg_sunspot_count
0,01-01-2011,47.0
1,02-01-2011,60.0
2,03-01-2011,75.0
3,04-01-2011,73.0
4,05-01-2011,60.0
...,...,...
115,08-01-2020,6.0
116,09-01-2020,7.0
117,10-01-2020,9.0
118,11-01-2020,8.0
